In [1]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

In [2]:
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "The dog barks at the quick brown fox.",
    "A lazy dog sleeps under the old tree.",
    "The fox runs into the dark forest.",
    "The brown dog chases the quick fox."
]

In [3]:
token = []

for sentence in corpus:
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-z\s]',"",sentence)
    split_sent = sentence.split()
    token.append(split_sent)

In [4]:
words = []

for sent in token:
    for w in sent:
        if w not in words:
            words.append(w)

In [5]:
word_to_index = {}
index_to_word = {}

for i in range(len(words)) :
    word_to_index[words[i]] = i
    index_to_word[i] = words[i]

print("Vocabulary: ", word_to_index)

Vocabulary:  {'the': 0, 'quick': 1, 'brown': 2, 'fox': 3, 'jumps': 4, 'over': 5, 'lazy': 6, 'dog': 7, 'barks': 8, 'at': 9, 'a': 10, 'sleeps': 11, 'under': 12, 'old': 13, 'tree': 14, 'runs': 15, 'into': 16, 'dark': 17, 'forest': 18, 'chases': 19}


In [6]:
voc_size = len(words)
window = 2
X= []
y= []

In [7]:
for sent in token:
    for i in range(window,len(sent)-window):
        context = sent[i-2:i] + sent[i+1:i+3]
        target = sent[i]

        context_indices = []
        for w in context:
              context_indices.append(word_to_index[w])
        X.append(context_indices)
        y.append(word_to_index[target])
X = np.array(X)
y= np.array(y)

In [8]:
print("Sample Context:", X[0])
print("Sample Target:", y[0])

Sample Context: [0 1 3 4]
Sample Target: 2


In [9]:
model = Sequential([
    Embedding(voc_size,10),
    GlobalAveragePooling1D(),
    Dense(voc_size,activation="softmax")
])

In [10]:
model.compile(
    optimizer="adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]
)

In [12]:
history = model.fit(X,y,epochs = 500, batch_size=8, verbose=0)

In [15]:
print("Training Loss:", history.history['loss'][-1])

Training Loss: 0.07718658447265625


In [18]:
embeddings = model.layers[0].get_weights()[0]

print("Word Embeddings:")

for word, index in word_to_index.items():
    print(word, embeddings[index])

Word Embeddings:
the [-0.9341639  -0.19735137  0.88357574 -0.16160545 -1.0382047   1.284555
  0.53642833  0.32862592  2.2427917   0.15109044]
quick [ 0.13726214 -1.1730056  -1.6683035   0.74487084 -0.30608884 -1.1125269
 -1.5277057   0.91662025 -1.2929041  -0.3704668 ]
brown [ 1.2673576   0.45517483 -0.17988431  0.12865764  1.9759692  -1.7526836
 -0.80856687 -0.45543474  1.2489737   0.5761534 ]
fox [ 2.1061907  -1.5989453   1.317723   -1.0141072  -0.10054472  0.3473408
  1.2587625   0.5637172  -0.82265615  0.21378058]
jumps [ 2.3250604  -0.11258968 -0.6613244  -0.00619783 -0.75635237  0.32335147
  0.54526025  0.7378641  -2.2288373   0.75137466]
over [ 1.8334324  -0.32920453  0.6527213   0.80340195  1.8334179  -1.5894839
 -1.9213734   0.47698066  0.04405164  0.73444027]
lazy [ 0.45013422  1.2145141  -0.47358656 -2.4332905  -0.26775062  0.69694644
 -0.23683989 -1.5635295  -0.04141472  0.06179241]
dog [-1.7578596  -0.4759827  -1.0371588   0.43652785 -0.43682358 -0.13083434
 -1.9071078  -0

In [21]:
context = ["brown","dog","the","quick"]
target = []

input = []
for w in context:
    input.append(word_to_index[w])
input = np.array([input])

predict = model.predict(input, verbose=0)
prediction = np.argmax(predict)

In [23]:
print("Context:", context)
print("Predicted Word:", index_to_word[prediction])
print()

Context: ['brown', 'dog', 'the', 'quick']
Predicted Word: chases

